# 02 - EDA & Visualisations

Nine publication-quality charts forming the narrative spine of the presentation.
Every chart saves a corresponding CSV for independent audit and reproduction.

| Chart | Type | Insight |
|---|---|---|
| 1 | Monthly revenue | Business scale and seasonality |
| 2 | Saturday dead zone | B2B identity proof (FULL raw source) |
| 3 | Guest vs Named AOV | High-value anonymous segment |
| 4 | International revenue | Geographic concentration |
| 5 | Hourly pattern | B2B business-hours signal |
| 6 | Top 15 products | Product concentration + outlier flags |
| 7 | Cancellation rate | Operational health |
| 8 | Cohort retention heatmap | **RESTORED** - lifecycle lifecycle lifecycle |
| 9 | Sankey basket flow | **RESTORED** - category purchase journey |

In [1]:
import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scipy.stats as stats

warnings.filterwarnings("ignore")

DATA_DIR   = "../outputs/data/"
CHARTS_DIR = "../outputs/charts/"
os.makedirs(CHARTS_DIR, exist_ok=True)

# ── Design system ─────────────────────────────────────────────────────────────
PALETTE = {
    "primary":   "#2E75B6",
    "accent":    "#E63946",
    "highlight": "#F4A261",
    "positive":  "#2A9D8F",
    "neutral":   "#8D99AE",
    "bg":        "#FAFAFA",
    "text":      "#1C1C1C",
    "grid":      "#E8E8E8",
}

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.facecolor":    PALETTE["bg"],
    "figure.facecolor":  PALETTE["bg"],
    "axes.grid":         True,
    "grid.color":        PALETTE["grid"],
    "grid.linestyle":    "--",
    "grid.alpha":        0.6,
    "font.size":         11,
    "axes.labelsize":    12,
    "axes.titlesize":    14,
    "axes.titleweight":  "bold",
})

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

# ── Load all intermediaries ───────────────────────────────────────────────────
monthly  = pd.read_csv(f"{DATA_DIR}01_monthly_summary.csv")
country  = pd.read_csv(f"{DATA_DIR}01_country_summary.csv")
hour     = pd.read_csv(f"{DATA_DIR}01_hour_of_day.csv")
products = pd.read_csv(f"{DATA_DIR}01_product_features.csv")
cancel   = pd.read_csv(f"{DATA_DIR}01_cancellation_rates.csv")
rfm      = pd.read_csv(f"{DATA_DIR}01_customer_rfm.csv")
txn      = pd.read_csv(f"{DATA_DIR}01_transactions_clean.csv",
                       parse_dates=["InvoiceDate"])
guest    = pd.read_csv(f"{DATA_DIR}01_guest_transactions.csv")
audit    = pd.read_csv(f"{DATA_DIR}01_audit_summary.csv")

# Saturday anomaly - MUST use full raw source (not post-clean)
dow_raw  = pd.read_csv(f"{DATA_DIR}01_day_of_week_raw.csv")
dow_post = pd.read_csv(f"{DATA_DIR}01_day_of_week.csv")

# Extract authoritative Saturday metadata from CSV
_sat_row       = dow_raw[dow_raw["DayOfWeek"] == "Saturday"].iloc[0]
SATURDAY_RATIO = float(_sat_row["saturday_ratio"])
SAT_ORDERS     = int(_sat_row["saturday_orders"])
WEEKDAY_MEAN   = float(_sat_row["weekday_mean_orders"])

print(f"Loaded. Saturday ratio (full raw): {SATURDAY_RATIO:.0f}:1")
print(f"  Transactions: {len(txn):,}  Customers: {rfm['CustomerID'].nunique():,}")

Loaded. Saturday ratio (full raw): 231:1
  Transactions: 820,221  Customers: 5,852


## Chart 1 - Monthly revenue trajectory

In [2]:
# ── Chart 1: Monthly Revenue Trajectory ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

x = range(len(monthly))
y = monthly["Revenue"] / 1e6

peak_months = [m for m in monthly["YearMonth"] if m.endswith(("-11", "-10", "-09"))]
bar_colors  = [
    PALETTE["accent"] if m in peak_months else PALETTE["primary"]
    for m in monthly["YearMonth"]
]

ax.bar(x, y, color=bar_colors, width=0.7, alpha=0.85, zorder=2)

z     = np.polyfit(list(x), y, 1)
trend = np.poly1d(z)
ax.plot(x, trend(list(x)), "--",
        color=PALETTE["neutral"], lw=1.5, alpha=0.7, label="Trend")

ax.set_xticks(list(x))
ax.set_xticklabels(monthly["YearMonth"], rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Revenue (GBP millions)")
ax.set_title("Monthly Revenue Trajectory")

peak_idx = int(y.idxmax())
peak_ym  = monthly["YearMonth"].iloc[peak_idx]
ax.annotate(
    f"Peak: GBP{y.iloc[peak_idx]:.2f}M ({peak_ym})",
    xy=(peak_idx, y.iloc[peak_idx]),
    xytext=(peak_idx - 2, y.iloc[peak_idx] + 0.12),
    arrowprops=dict(arrowstyle="->", color=PALETTE["text"]),
    fontsize=9, color=PALETTE["text"],
)
ax.legend(handles=[
    mpatches.Patch(color=PALETTE["accent"],  label="Sep/Oct/Nov (holiday ramp)"),
    mpatches.Patch(color=PALETTE["primary"], label="Other months"),
], loc="upper left", fontsize=9)

fig.tight_layout()
path = f"{CHARTS_DIR}chart_02_01_monthly_revenue.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path}")

# CSV export
monthly_exp = monthly.copy()
monthly_exp["Revenue_millions"] = monthly_exp["Revenue"] / 1e6
monthly_exp["trend_fitted"]     = trend(list(x))
monthly_exp["is_peak_month"]    = monthly_exp["YearMonth"].str.endswith(("-11", "-10", "-09"))
monthly_exp.to_csv(f"{DATA_DIR}02_chart01_monthly_revenue.csv", index=False)
print(f"  CSV: 02_chart01_monthly_revenue.csv")

  Saved: ../outputs/charts/chart_02_01_monthly_revenue.png
  CSV: 02_chart01_monthly_revenue.csv


## Chart 2 - Saturday dead zone (B2B proof)

**Sources are bifurcated by design:**
- Panel A (Orders) → `01_day_of_week_raw.csv` (full raw 1M+ rows, authoritative)
- Panel B (Revenue) → `01_day_of_week.csv` (post-clean basket totals, financial EDA)

In [3]:
# ── Chart 2: Saturday Dead Zone - FULL RAW SOURCE ─────────────────────────────
# Panel A (Orders): dow_raw   - full raw 1M+ rows, authoritative ratio
# Panel B (Revenue): dow_post - post-clean basket totals, financial consistency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dow_raw_sorted  = dow_raw.copy()
dow_post_sorted = dow_post.copy()
for df_d in [dow_raw_sorted, dow_post_sorted]:
    df_d["DayOfWeek"] = pd.Categorical(
        df_d["DayOfWeek"], categories=DAY_ORDER, ordered=True
    )
dow_raw_sorted  = dow_raw_sorted.sort_values("DayOfWeek").reset_index(drop=True)
dow_post_sorted = dow_post_sorted.sort_values("DayOfWeek").reset_index(drop=True)

sat_idx = list(dow_raw_sorted["DayOfWeek"]).index("Saturday")

# Panel A - Orders (full raw)
colors_a = [
    PALETTE["accent"] if d == "Saturday" else PALETTE["primary"]
    for d in dow_raw_sorted["DayOfWeek"]
]
axes[0].bar(dow_raw_sorted["DayOfWeek"], dow_raw_sorted["Orders"],
            color=colors_a, width=0.6, alpha=0.85, zorder=2)
axes[0].set_title(
    f"Orders by Day of Week\n(source: FULL raw dataset - all {len(dow_raw_sorted)} rows)",
    pad=10,
)
axes[0].set_ylabel("Distinct Orders")
axes[0].tick_params(axis="x", rotation=30)
axes[0].annotate(
    f"{SAT_ORDERS:,} orders\n{SATURDAY_RATIO:.0f}:1 ratio",
    xy=(sat_idx, SAT_ORDERS),
    xytext=(sat_idx - 1.6, SAT_ORDERS + dow_raw_sorted["Orders"].max() * 0.13),
    arrowprops=dict(arrowstyle="->", color=PALETTE["accent"]),
    fontsize=9, color=PALETTE["accent"], fontweight="bold",
)

# Panel B - Revenue (post-clean)
colors_b = [
    PALETTE["accent"] if d == "Saturday" else PALETTE["positive"]
    for d in dow_post_sorted["DayOfWeek"]
]
axes[1].bar(dow_post_sorted["DayOfWeek"], dow_post_sorted["Revenue"] / 1e6,
            color=colors_b, width=0.6, alpha=0.85, zorder=2)
axes[1].set_title(
    "Revenue by Day of Week (GBP M)\n(source: post-clean basket totals)",
    pad=10,
)
axes[1].set_ylabel("Revenue (GBP millions)")
axes[1].tick_params(axis="x", rotation=30)

fig.suptitle(
    f"B2B Identity Proof: Saturday Dead Zone  ({SATURDAY_RATIO:.0f}:1 ratio)",
    fontsize=14, fontweight="bold", y=1.02,
)
fig.tight_layout()
path = f"{CHARTS_DIR}chart_02_02_saturday_anomaly.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path}")

# CSV export
dow_exp = dow_raw_sorted[
    ["DayOfWeek", "Orders", "saturday_orders", "weekday_mean_orders",
     "saturday_ratio", "source"]
].copy()
dow_exp = dow_exp.merge(
    dow_post_sorted[["DayOfWeek", "Revenue"]].rename(
        columns={"Revenue": "Revenue_postclean"}
    ),
    on="DayOfWeek", how="left",
)
dow_exp["is_saturday"]    = dow_exp["DayOfWeek"] == "Saturday"
dow_exp["orders_source"]  = "01_day_of_week_raw.csv (full raw)"
dow_exp["revenue_source"] = "01_day_of_week.csv (post-clean)"
dow_exp.to_csv(f"{DATA_DIR}02_chart02_saturday_dead_zone.csv", index=False)
print(f"  CSV: 02_chart02_saturday_dead_zone.csv  ratio={SATURDAY_RATIO:.0f}:1")

  Saved: ../outputs/charts/chart_02_02_saturday_anomaly.png
  CSV: 02_chart02_saturday_dead_zone.csv  ratio=231:1


## Chart 3 - Guest vs Named AOV

In [4]:
# ── Chart 3: Guest vs Named AOV ───────────────────────────────────────────────
guest_pos = guest[
    (guest["Quantity"] > 0) &
    (guest["Price"]    > 0) &
    (~guest["Invoice"].str.startswith("C", na=False))
].copy()
guest_pos["Revenue"] = guest_pos["Quantity"] * guest_pos["Price"]
g_inv = guest_pos.groupby("Invoice")["Revenue"].sum()

named_pos = txn[(txn["Quantity"] > 0) & (~txn["IsCancelled"])].copy()
n_inv     = named_pos.groupby("Invoice")["Revenue"].sum()

mwu_stat, mwu_p = stats.mannwhitneyu(g_inv, n_inv, alternative="two-sided")
r_effect = 1 - (2 * mwu_stat) / (len(g_inv) * len(n_inv))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: bar comparison
labels     = ["Guest\n(Anonymous)", "Named\n(Identified)"]
means      = [g_inv.mean(), n_inv.mean()]
sems       = [g_inv.sem(), n_inv.sem()]
axes[0].bar(labels, means,
            color=[PALETTE["accent"], PALETTE["primary"]],
            width=0.5, alpha=0.85,
            yerr=sems, capsize=6,
            error_kw={"color": PALETTE["neutral"]}, zorder=2)
axes[0].set_title("Average Order Value by Customer Type")
axes[0].set_ylabel("Mean Invoice Value (GBP)")
for i, m in enumerate(means):
    axes[0].text(i, m + 30, f"GBP{m:,.0f}", ha="center",
                 fontweight="bold", fontsize=11)
axes[0].text(0.5, max(means) * 0.55,
             f"p = {mwu_p:.2e} (Mann-Whitney U)\nr = {r_effect:.3f}",
             ha="center", fontsize=9, color=PALETTE["neutral"], style="italic",
             transform=axes[0].transData)

# Panel B: revenue split pie
total_rev   = g_inv.sum() + n_inv.sum()
axes[1].pie(
    [g_inv.sum() / total_rev, n_inv.sum() / total_rev],
    labels=[
        f"Guest GBP{g_inv.sum()/1e6:.1f}M ({g_inv.sum()/total_rev:.1%})",
        f"Named GBP{n_inv.sum()/1e6:.1f}M ({n_inv.sum()/total_rev:.1%})",
    ],
    colors=[PALETTE["accent"], PALETTE["primary"]],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 10},
)
axes[1].set_title("Revenue Split: Guest vs Named")

fig.suptitle(
    f"Guest AOV = GBP{g_inv.mean():,.0f} vs Named AOV = GBP{n_inv.mean():,.0f}  "
    f"(p = {mwu_p:.2e})",
    fontsize=13, fontweight="bold",
)
fig.tight_layout()
path = f"{CHARTS_DIR}chart_02_03_guest_aov.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path}")

# CSV export
pd.DataFrame({
    "customer_type":      ["Guest (Anonymous)", "Named (Identified)"],
    "n_invoices":         [len(g_inv), len(n_inv)],
    "mean_invoice_gbp":   [g_inv.mean(), n_inv.mean()],
    "median_invoice_gbp": [g_inv.median(), n_inv.median()],
    "sem_gbp":            [g_inv.sem(), n_inv.sem()],
    "total_revenue_gbp":  [g_inv.sum(), n_inv.sum()],
    "revenue_share_pct":  [g_inv.sum() / total_rev * 100, n_inv.sum() / total_rev * 100],
    "mwu_stat":           [mwu_stat, mwu_stat],
    "mwu_p_value":        [mwu_p, mwu_p],
    "effect_size_r":      [r_effect, r_effect],
}).to_csv(f"{DATA_DIR}02_chart03_guest_named_aov.csv", index=False)
print("  CSV: 02_chart03_guest_named_aov.csv")

  Saved: ../outputs/charts/chart_02_03_guest_aov.png
  CSV: 02_chart03_guest_named_aov.csv


## Chart 4 - International revenue

In [5]:
# ── Chart 4: International Revenue ────────────────────────────────────────────
intl = (
    country[country["Country"] != "United Kingdom"]
    .nlargest(12, "Revenue")
    .sort_values("Revenue")
)

fig, ax = plt.subplots(figsize=(11, 6))
bar_colors = [
    PALETTE["highlight"] if i >= len(intl) - 3 else PALETTE["primary"]
    for i in range(len(intl))
]
bars = ax.barh(intl["Country"], intl["Revenue"] / 1e3,
               color=bar_colors, alpha=0.85, zorder=2)

for bar, (_, row) in zip(bars, intl.iterrows()):
    ax.text(bar.get_width() + 2,
            bar.get_y() + bar.get_height() / 2,
            f"GBP{row['Revenue']/1e3:.1f}K  ({row['RevenueShare']:.1%})",
            va="center", fontsize=9)

ax.set_xlabel("Revenue (GBP thousands)")
ax.set_title("Top 12 International Markets (excl. UK)")
uk_row = country[country["Country"] == "United Kingdom"].iloc[0]
fig.text(0.5, -0.04,
         f"UK: GBP{uk_row['Revenue']/1e6:.1f}M ({uk_row['RevenueShare']:.1%}) - "
         "B2B centralised wholesale distribution model",
         ha="center", fontsize=9, style="italic", color=PALETTE["neutral"])

fig.tight_layout()
path = f"{CHARTS_DIR}chart_02_04_international_revenue.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path}")
country.to_csv(f"{DATA_DIR}02_chart04_country_revenue.csv", index=False)
print("  CSV: 02_chart04_country_revenue.csv")

  Saved: ../outputs/charts/chart_02_04_international_revenue.png
  CSV: 02_chart04_country_revenue.csv


## Chart 5 - Hourly trading pattern

In [6]:
# ── Chart 5: Hourly Trading Pattern ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
peak_hours = [10, 11, 12, 13, 14]
bar_colors = [
    PALETTE["accent"] if h in peak_hours else PALETTE["primary"]
    for h in hour["Hour"]
]
ax.bar(hour["Hour"], hour["Orders"], color=bar_colors, width=0.7,
       alpha=0.85, zorder=2)
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Distinct Orders")
ax.set_title("Order Volume by Hour - Business Hours Signal")
ax.set_xticks(range(0, 24))

peak_h_orders = int(hour.loc[hour["Hour"] == 12, "Orders"].values[0])
ax.annotate(
    f"Peak 10:00-14:00\n(B2B business hours)\n{peak_h_orders:,} @ 12:00",
    xy=(12, peak_h_orders),
    xytext=(16, hour["Orders"].max() * 0.85),
    arrowprops=dict(arrowstyle="->", color=PALETTE["text"]),
    fontsize=9,
)
ax.axvspan(10, 14, alpha=0.1, color=PALETTE["accent"], zorder=1)

fig.tight_layout()
path = f"{CHARTS_DIR}chart_02_05_hourly_pattern.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path}")
hour.to_csv(f"{DATA_DIR}02_chart05_hourly_orders.csv", index=False)
print("  CSV: 02_chart05_hourly_orders.csv")

  Saved: ../outputs/charts/chart_02_05_hourly_pattern.png
  CSV: 02_chart05_hourly_orders.csv


## Chart 6 - Top 15 products by revenue

In [7]:
# ── Chart 6: Top 15 Products by Revenue ───────────────────────────────────────
top15 = products.head(15).sort_values("TotalRevenue").copy()
top15["Label"] = top15["Description"].str[:35]

fig, ax = plt.subplots(figsize=(12, 7))
bar_colors = [
    PALETTE["accent"] if row.get("IsSingleCustomerProduct", 0) == 1
    else (PALETTE["highlight"] if i >= 12 else PALETTE["primary"])
    for i, (_, row) in enumerate(top15.iterrows())
]
bars = ax.barh(top15["Label"], top15["TotalRevenue"] / 1e3,
               color=bar_colors, alpha=0.85)

for bar, (_, row) in zip(bars, top15.iterrows()):
    ax.text(bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f"GBP{row['TotalRevenue']/1e3:.1f}K  ({int(row['NumCustomers'])} custs)",
            va="center", fontsize=8)

ax.set_xlabel("Revenue (GBP thousands)")
ax.set_title("Top 15 Products by Revenue")

handles = [
    mpatches.Patch(color=PALETTE["accent"],    label="Single-customer (outlier)"),
    mpatches.Patch(color=PALETTE["highlight"], label="Top-3 product"),
    mpatches.Patch(color=PALETTE["primary"],   label="Other top-15"),
]
ax.legend(handles=handles, fontsize=9, loc="lower right")

fig.tight_layout()
path = f"{CHARTS_DIR}chart_02_06_top_products.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path}")
top15.to_csv(f"{DATA_DIR}02_chart06_top_products.csv", index=False)
print("  CSV: 02_chart06_top_products.csv")

  Saved: ../outputs/charts/chart_02_06_top_products.png
  CSV: 02_chart06_top_products.csv


## Chart 7 - Monthly cancellation rate

In [8]:
# ── Chart 7: Monthly Cancellation Rate ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(cancel))
y = cancel["CancelRate"] * 100
mean_c = y.mean()

ax.fill_between(x, y, alpha=0.2, color=PALETTE["accent"])
ax.plot(x, y, color=PALETTE["accent"], lw=2, marker="o", markersize=4, zorder=3)
ax.axhline(mean_c, linestyle="--", color=PALETTE["neutral"], lw=1.5,
           label=f"Mean {mean_c:.1f}%")

ax.set_xticks(list(x))
ax.set_xticklabels(cancel["YearMonth"], rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Cancellation Rate (%)")
ax.set_title(
    f"Monthly Invoice Cancellation Rate  "
    f"(mean = {mean_c:.1f}%  |  B2B stock-revision signal)"
)
ax.legend(fontsize=9)

fig.tight_layout()
path = f"{CHARTS_DIR}chart_02_07_cancellation_rate.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path}")
cancel.to_csv(f"{DATA_DIR}02_chart07_cancellation_rate.csv", index=False)
print("  CSV: 02_chart07_cancellation_rate.csv")

  Saved: ../outputs/charts/chart_02_07_cancellation_rate.png
  CSV: 02_chart07_cancellation_rate.csv


## Chart 8 - Monthly cohort retention heatmap

Restored from the old pipeline.  Uses the **full transaction table** - no head() or
subset.  Shows month-0 to month-12 retention rates for every acquisition cohort.

In [9]:
# ── Chart 8: Monthly Cohort Retention Heatmap ─────────────────────────────────
# Full transaction table - no subsets.
import seaborn as sns

txn["CohortMonth"] = (
    txn.groupby("CustomerID")["InvoiceDate"]
    .transform("min").dt.to_period("M").astype(str)
)
txn["OrderMonth"] = txn["InvoiceDate"].dt.to_period("M").astype(str)

cohort = (
    txn.groupby(["CohortMonth", "OrderMonth"])["CustomerID"]
    .nunique()
    .reset_index()
)
cohort.columns = ["CohortMonth", "OrderMonth", "Customers"]

cohort["CM_dt"] = pd.to_datetime(cohort["CohortMonth"])
cohort["OM_dt"] = pd.to_datetime(cohort["OrderMonth"])
cohort["MonthNumber"] = (
    (cohort["OM_dt"].dt.year  - cohort["CM_dt"].dt.year)  * 12
    + (cohort["OM_dt"].dt.month - cohort["CM_dt"].dt.month)
)

cohort_sizes = (
    cohort[cohort["MonthNumber"] == 0]
    .set_index("CohortMonth")["Customers"]
)
cohort["CohortSize"]    = cohort["CohortMonth"].map(cohort_sizes)
cohort["RetentionRate"] = (cohort["Customers"] / cohort["CohortSize"]).round(4)

cohort.to_csv(f"{DATA_DIR}02_chart08_cohort_retention.csv", index=False)
print("  CSV: 02_chart08_cohort_retention.csv")

pivot = (
    cohort[cohort["MonthNumber"].between(0, 12)]
    .pivot_table(
        index="CohortMonth",
        columns="MonthNumber",
        values="RetentionRate",
        aggfunc="mean",
    )
    .head(20)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    pivot * 100,
    annot=True,
    fmt=".0f",
    linewidths=0.4,
    cmap="Blues",
    ax=ax,
    cbar_kws={"label": "Retention Rate (%)", "shrink": 0.6},
    annot_kws={"size": 8},
)
ax.set_title(
    "Customer Cohort Retention Matrix  (Monthly Cohorts, %)",
    fontsize=13, pad=14,
)
ax.set_xlabel("Months Since First Purchase")
ax.set_ylabel("Acquisition Cohort")

fig.tight_layout()
path = f"{CHARTS_DIR}chart_02_08_cohort_heatmap.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path}")

  CSV: 02_chart08_cohort_retention.csv
  Saved: ../outputs/charts/chart_02_08_cohort_heatmap.png


## Chart 9 - Sankey basket flow

Restored from the old pipeline.  Requires the **Category column** built by
`categorise()` in NB01.  Shows how customers move between macro-categories
from their 1st to 2nd to 3rd basket.

In [10]:
# ── Chart 9: Sankey Basket Flow Diagram ───────────────────────────────────────
# Customer journey: category of 1st -> 2nd -> 3rd basket.
# Uses macro-Category column from NB01 - full dataset, no subset.

inv_cats = (
    txn.groupby(["CustomerID", "Invoice"])
    .agg(
        Category    = ("Category",    lambda x: x.mode()[0]),
        InvoiceDate = ("InvoiceDate", "min"),
    )
    .reset_index()
    .sort_values(["CustomerID", "InvoiceDate"])
)
inv_cats["PurchaseNum"] = inv_cats.groupby("CustomerID").cumcount() + 1

p1 = (inv_cats[inv_cats["PurchaseNum"] == 1][["CustomerID", "Category"]]
      .rename(columns={"Category": "Cat1"}))
p2 = (inv_cats[inv_cats["PurchaseNum"] == 2][["CustomerID", "Category"]]
      .rename(columns={"Category": "Cat2"}))
p3 = (inv_cats[inv_cats["PurchaseNum"] == 3][["CustomerID", "Category"]]
      .rename(columns={"Category": "Cat3"}))

journeys = p1.merge(p2, on="CustomerID").merge(p3, on="CustomerID")
print(f"  Customer journeys (3+ purchases): {len(journeys):,}")

cats = sorted(txn["Category"].unique())
node_labels = (
    [f"1st: {c}" for c in cats]
    + [f"2nd: {c}" for c in cats]
    + [f"3rd: {c}" for c in cats]
)

src_nodes, tgt_nodes, vals = [], [], []

for _, row in (
    journeys.groupby(["Cat1", "Cat2"]).size().reset_index(name="n").iterrows()
):
    if row["n"] >= 5:
        src_nodes.append(cats.index(row["Cat1"]))
        tgt_nodes.append(len(cats) + cats.index(row["Cat2"]))
        vals.append(row["n"])

for _, row in (
    journeys.groupby(["Cat2", "Cat3"]).size().reset_index(name="n").iterrows()
):
    if row["n"] >= 5:
        src_nodes.append(len(cats) + cats.index(row["Cat2"]))
        tgt_nodes.append(2 * len(cats) + cats.index(row["Cat3"]))
        vals.append(row["n"])

NODE_COLORS = [
    "#2E75B6", "#2A9D8F", "#F4A261", "#E63946",
    "#8D99AE", "#9B59B6", "#1ABC9C", "#E74C3C",
    "#3498DB", "#E67E22",
]
node_colors = (NODE_COLORS * 5)[:len(node_labels)]

fig_sankey = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(pad=20, thickness=18, label=node_labels, color=node_colors),
    link=dict(source=src_nodes, target=tgt_nodes, value=vals,
              color="rgba(46,117,182,0.25)"),
))
fig_sankey.update_layout(
    title_text="Customer Purchase Journey - 1st to 3rd Basket Category Flow",
    paper_bgcolor=PALETTE["bg"],
    height=560, font_size=11,
)

path = f"{CHARTS_DIR}chart_02_09_sankey_basket_flow.png"
fig_sankey.write_image(path, scale=2, width=1200, height=600)
print(f"  Saved: {path}")

journeys.to_csv(f"{DATA_DIR}02_chart09_customer_journeys.csv", index=False)
print("  CSV: 02_chart09_customer_journeys.csv")
print(f"  Sankey: {len(journeys):,} journeys across {len(cats)} categories")

  Customer journeys (3+ purchases): 3,562
  Saved: ../outputs/charts/chart_02_09_sankey_basket_flow.png
  CSV: 02_chart09_customer_journeys.csv
  Sankey: 3,562 journeys across 10 categories


## Final output summary

In [11]:
# ── Final summary ─────────────────────────────────────────────────────────────
chart_files = sorted([f for f in os.listdir(CHARTS_DIR) if "chart_02" in f])
data_files  = sorted([f for f in os.listdir(DATA_DIR)   if f.startswith("02_")])

print(f"Notebook 02 complete")
print(f"  Charts saved: {len(chart_files)}")
for c in chart_files:
    kb = os.path.getsize(f"{CHARTS_DIR}{c}") // 1024
    print(f"    {c:<55} {kb:>5} KB")

print(f"  CSVs saved: {len(data_files)}")
for d in data_files:
    kb = os.path.getsize(f"{DATA_DIR}{d}") // 1024
    print(f"    {d:<55} {kb:>5} KB")

Notebook 02 complete
  Charts saved: 9
    chart_02_01_monthly_revenue.png                            77 KB
    chart_02_02_saturday_anomaly.png                          113 KB
    chart_02_03_guest_aov.png                                  96 KB
    chart_02_04_international_revenue.png                     102 KB
    chart_02_05_hourly_pattern.png                             67 KB
    chart_02_06_top_products.png                              157 KB
    chart_02_07_cancellation_rate.png                          97 KB
    chart_02_08_cohort_heatmap.png                            240 KB
    chart_02_09_sankey_basket_flow.png                       1117 KB
  CSVs saved: 9
    02_chart01_monthly_revenue.csv                              1 KB
    02_chart02_saturday_dead_zone.csv                           1 KB
    02_chart03_guest_named_aov.csv                              0 KB
    02_chart04_country_revenue.csv                              1 KB
    02_chart05_hourly_orders.csv                